In [ ]:
import requests
import json
import os
from datetime import datetime

# Predefined offline rates
offline_rates = {
    "USD": {"EUR": 0.85, "NGN": 150, "GBP": 0.75},
    "EUR": {"USD": 1.17, "NGN": 176, "GBP": 0.88}
}

CACHE_FILE = "exchange_cache.json"
HISTORY_FILE = "conversion_history.json"

# Function to fetch exchange rates based on base currency
def get_exchange_rates(base_currency):
    try:
        # Check if exchange rates are cached
        if os.path.exists(CACHE_FILE):
            with open(CACHE_FILE, 'r') as file:
                cache = json.load(file)
                if base_currency in cache:
                    return cache[base_currency]

        # Send GET request to API
        url = f"https://api.exchangerate-api.com/v4/latest/{base_currency}"
        response = requests.get(url)
        data = response.json()

        # Save to cache
        cache = {}
        if os.path.exists(CACHE_FILE):
            with open(CACHE_FILE, 'r') as file:
                cache = json.load(file)
        cache[base_currency] = data['rates']
        with open(CACHE_FILE, 'w') as file:
            json.dump(cache, file)

        return data['rates']
    except:
        # Return fallback rates if request fails
        return offline_rates.get(base_currency, {})

# Function to save each conversion to history
def save_conversion_history(base, amount, results):
    history = []
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as file:
            history = json.load(file)

    history.append({
        "timestamp": str(datetime.now()),
        "base_currency": base,
        "amount": amount,
        "results": results
    })

    with open(HISTORY_FILE, 'w') as file:
        json.dump(history, file, indent=2)

# Get user input
base_currency = input("Enter base currency (e.g., USD, EUR): ").upper()
target_currencies = input("Enter target currencies (comma-separated): ").upper().split(",")
amount = float(input("Enter amount: "))

# Fetch exchange rates for the base currency
rates = get_exchange_rates(base_currency)

# Perform conversion if target currency is valid
results = {}
if rates:
    for target_currency in target_currencies:
        target_currency = target_currency.strip()
        if target_currency in rates:
            converted_amount = amount * rates[target_currency]
            results[target_currency] = round(converted_amount, 2)
            print(f"{amount} {base_currency} is {converted_amount:.2f} {target_currency}")
        else:
            print(f"Invalid currency code: {target_currency}")
    if results:
        save_conversion_history(base_currency, amount, results)
else:
    print("Could not fetch exchange rates. Check internet or try again later.")

# Optionally show recent history
view = input("View last 5 conversions? (yes/no): ").lower()
if view == "yes" and os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, 'r') as file:
        history = json.load(file)
        print("\nRecent Conversions:")
        for entry in history[-5:]:
            print(f"{entry['timestamp']} — {entry['amount']} {entry['base_currency']} ➡ {entry['results']}")